# 01 · Scraping

Descarga desde iNaturalist a `Insectos/<especie>/` y registra la procedencia en
`metadata.csv`. Incremental: deduplica por `photo_id`, así que reejecutar solo
baja lo que falta.

Solo licencias abiertas. Quedan fuera las fotos sin licencia declarada y
`cc-by-nc-nd`, cuyo ND prohíbe obras derivadas — recortar y aumentar lo es.

In [ ]:
import hashlib
import json
import re
import time

import requests

from corpus import IMAGENES, LISTA_ESPECIES, METADATA, escribirCsv, leerCsv

API = "https://api.inaturalist.org/v1"
LICENCIAS = "cc0,cc-by,cc-by-nc,cc-by-sa,cc-by-nc-sa"
LICENCIAS_OK = set(LICENCIAS.split(","))
PAUSA = 0.25          # entre descargas de fotos
PAUSA_API = 1.0       # entre llamadas a la API

POR_ESPECIE = 350
SOLO_ESPECIES = None          # {"eurybia_dardus"} para rellenar unas pocas
INCLUIR_NO_CONFIRMADAS = False

COLUMNAS = ["especie", "taxon_id", "observation_id", "photo_id", "quality_grade",
            "licencia", "atribucion", "url", "archivo", "sha256"]

## Cliente

In [2]:
def pedir(sesion, ruta, params, intentos=5):
    """Reintenta con espera creciente ante 429 y cortes de red."""
    for intento in range(intentos):
        try:
            respuesta = sesion.get(f"{API}/{ruta}", params=params, timeout=30)
        except requests.RequestException as e:
            print(f"    {type(e).__name__}, reintento en {2 ** intento}s")
            time.sleep(2 ** intento)
            continue
        if respuesta.status_code == 429:
            espera = float(respuesta.headers.get("Retry-After", 2 ** intento))
            print(f"    429, esperando {espera:.0f}s")
            time.sleep(espera)
            continue
        respuesta.raise_for_status()
        return respuesta.json()
    raise RuntimeError(f"{ruta} falló tras {intentos} intentos")


def nombreCorto(entrada):
    limpio = re.sub(r"\(.*?\)|,.*$", "", entrada)
    return " ".join(limpio.split()[:2])


def carpetaDe(nombre):
    return nombre.lower().replace(" ", "_")


def buscarTaxon(sesion, nombre):
    resultados = pedir(sesion, "taxa", {"q": nombre, "rank": "species",
                                        "per_page": 1, "is_active": "true"})["results"]
    return resultados[0] if resultados else None


def disponibilidad(sesion, taxonId):
    """Techo real de una especie, para saber si subir el tope sirve de algo."""
    return pedir(sesion, "observations", {
        "taxon_id": taxonId, "quality_grade": "research", "photos": "true",
        "photo_license": LICENCIAS, "per_page": 1,
    })["total_results"]


def descargar(sesion, url, destino):
    respuesta = sesion.get(url, timeout=45)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    return hashlib.sha256(respuesta.content).hexdigest()

## Paginación

`incluirNoConfirmadas` quita el filtro `quality_grade`: es la única fuente que
hace crecer a las especies escasas, que ya agotaron lo research-grade, a cambio
de identificaciones sin confirmar. Cada foto sale con su grado para poder
separarlas después.

In [3]:
def fotosDe(sesion, taxonId, maximo, incluirNoConfirmadas=False):
    vistas, pagina = set(), 1
    while len(vistas) < maximo:
        filtros = {"taxon_id": taxonId, "photos": "true", "photo_license": LICENCIAS,
                   "order_by": "id", "order": "asc", "per_page": 200, "page": pagina}
        if not incluirNoConfirmadas:
            filtros["quality_grade"] = "research"

        observaciones = pedir(sesion, "observations", filtros)["results"]
        if not observaciones:
            return
        for observacion in observaciones:
            for foto in observacion["photos"]:
                # el filtro del servidor es por observación: una válida arrastra
                # sus otras fotos, que pueden ser ND
                if foto["id"] in vistas or foto.get("license_code") not in LICENCIAS_OK:
                    continue
                vistas.add(foto["id"])
                yield observacion["id"], observacion.get("quality_grade", ""), foto
                if len(vistas) >= maximo:
                    return
        pagina += 1

## Descarga

Acumula sobre `metadata.csv` y lo reescribe entero al final

In [6]:
especies = json.loads(LISTA_ESPECIES.read_text())["species"]
especies = [e for e in especies if "sp." not in e["name"]]
if SOLO_ESPECIES:
    especies = [e for e in especies if carpetaDe(nombreCorto(e["name"])) in SOLO_ESPECIES]

sesion = requests.Session()
sesion.headers["User-Agent"] = "butterfly-dataset/1.0"

filas = leerCsv(METADATA)
caidas = []
conocidas = {fila["photo_id"] for fila in filas}
print(f"{len(filas)} fotos ya registradas · {len(especies)} especies a recorrer")

for numero, entrada in enumerate(especies, start=1):
    nombre = nombreCorto(entrada["name"])
    taxonId = entrada.get("taxon_id")
    if taxonId is None:
        taxon = buscarTaxon(sesion, nombre)
        time.sleep(PAUSA)
        if taxon is None:
            print(f"[{numero}/{len(especies)}] {nombre}: sin taxón")
            continue
        taxonId = taxon["id"]

    carpeta = IMAGENES / carpetaDe(nombre)
    carpeta.mkdir(parents=True, exist_ok=True)
    nuevas = 0
    for observacionId, grado, foto in fotosDe(sesion, taxonId, POR_ESPECIE,
                                              INCLUIR_NO_CONFIRMADAS):
        url = foto["url"].replace("square", "large")
        archivo = carpeta / f"{carpetaDe(nombre)}_{foto['id']}.jpg"
        if str(foto["id"]) in conocidas and archivo.exists():
            continue
        try:
            sha = descargar(sesion, url, archivo)
        except requests.RequestException as e:   # una foto caída no debe abortar la corrida
            caidas.append({"archivo": archivo.name, "url": url, "error": str(e)[:80]})
            time.sleep(PAUSA)
            continue
        filas.append({
            "especie": carpetaDe(nombre), "taxon_id": taxonId,
            "observation_id": observacionId, "photo_id": foto["id"],
            "quality_grade": grado, "licencia": foto["license_code"],
            "atribucion": foto.get("attribution", ""), "url": url,
            "archivo": archivo.name, "sha256": sha,
        })
        nuevas += 1
        time.sleep(PAUSA)

    filas = list({f["photo_id"]: f for f in filas}.values())
    escribirCsv(METADATA, filas, COLUMNAS)   # por especie: interrumpir no deja imágenes sin procedencia
    print(f"[{numero}/{len(especies)}] {nombre}: +{nuevas}")

print(f"\n{len(filas)} fotos en metadata.csv · {len(caidas)} descargas fallidas")

43745 fotos ya registradas · 100 especies a recorrer
[1/100] Anartia amathea: +0
[2/100] Heliconius doris: +0
[3/100] Anartia jatrophae: +0
[4/100] Heraclides thoas: +0
[5/100] Hamadryas feronia: +0
[6/100] Siproeta stelenes: +0
[7/100] Melanis electron: +0
[8/100] Rhetus periander: +0
[9/100] Hamadryas amphinome: +0
[10/100] Colobura dirce: +0
[11/100] Pseudohaetera hypaesia: +0
[12/100] Siproeta epaphus: +0
[13/100] Heliconius eleuchia: +0
[14/100] Dryas iulia: +0
[15/100] Dione moneta: +0
[16/100] Euptychoides laccine: +0
[17/100] Historis odius: +0
[18/100] Chlosyne lacinia: +0
[19/100] Heraclides anchisiades: +0
[20/100] Danaus plexippus: +0
[21/100] Battus polydamas: +0
[22/100] Dione vanillae: +0
[23/100] Dione juno: +0
[24/100] Anteos clorinde: +0
[25/100] Hamadryas laodamia: +0
[26/100] Dione glycera: +0
[27/100] Heliconius clysonymus: +0
[28/100] Callicore pitheas: +0
[29/100] Adelpha alala: +0
[30/100] Hemiargus hanno: +0
[31/100] Parides eurimedes: +0
[32/100] Hamadryas feb

## Inventario

In [ ]:
import pandas as pd

enDisco = {(carpeta.name, foto.name)
           for carpeta in IMAGENES.iterdir() if carpeta.is_dir()
           for foto in carpeta.glob("*.jpg")}

meta = pd.DataFrame(filas).drop_duplicates("photo_id")
meta = meta[[(e, a) in enDisco for e, a in zip(meta.especie, meta.archivo)]]
porEspecie = meta.groupby("especie").agg(fotos=("photo_id", "size"),
                                         observaciones=("observation_id", "nunique"))
print(f"{len(porEspecie)} especies · {porEspecie.fotos.sum():,} fotos")
print(f"por especie: min {porEspecie.fotos.min()}, mediana "
      f"{porEspecie.fotos.median():.0f}, max {porEspecie.fotos.max()}")
porEspecie.sort_values("fotos").head(10)

100 especies · 35,854 fotos
por especie: min 287, mediana 350, max 700


,fotos,observaciones
especie,,
epargyreus_cruza,287,158
eueides_procula,304,191
mesosemia_mevania,315,186
actinote_neleus,329,203
caligo_eurilochus,332,211
taygetis_mermeria,338,256
dione_moneta,349,208
marpesia_berania,350,223
manataria_maculata,350,246


## Cuánto más hay disponible

Antes de subir el tope conviene saber el techo: una especie que ya agotó
iNaturalist no crece por más que se scrapee.

In [8]:
techos = {}
for especie, fila in porEspecie.iterrows():
    taxon = meta[meta.especie == especie].taxon_id.iloc[0]
    techos[especie] = disponibilidad(sesion, int(taxon))
    time.sleep(PAUSA_API)

porEspecie["obs_disponibles"] = pd.Series(techos)
porEspecie["fotos_estimadas"] = (porEspecie.obs_disponibles
                                 * porEspecie.fotos / porEspecie.observaciones).round()
for tope in (200, 350, 500):
    n = (porEspecie.fotos_estimadas >= tope).sum()
    print(f"  alcanzarían {tope} fotos: {n}/{len(porEspecie)} especies")
porEspecie.nsmallest(10, "fotos_estimadas")

KeyboardInterrupt: 